# RF-DETR — two-stage transfer: real flat-bug pretrain, then synthetic

**Why:** flatbug beats RF-DETR mainly on *localisation* (mAP@75 0.468 vs 0.340 on the 20k dataset), and flatbug's backbone was pretrained on real arthropod images. This notebook gives RF-DETR the same advantage: **Stage 1** learns to *find* insects from real photographs (single class), **Stage 2** learns to *classify* the 23 orders from the synthetic trays.

**Baseline to beat:** single-stage RF-DETR on the same 20k dataset = **0.3476** mAP@50:95 (mAP@50 0.641, mAP@75 0.340).

---

### Prerequisite — download and prepare the flat-bug data (once, in a terminal)

The dataset is openly available on Zenodo (6.9 GB, 6,028 real arthropod images, 23 sub-datasets, COCO annotations, no credentials needed):

```bash
wget -c -O flatbug-dataset.zip \
  "https://zenodo.org/api/records/14761447/files/flatbug-dataset.zip/content"
mkdir -p pre-pro && unzip -q flatbug-dataset.zip -d pre-pro

# each sub-directory must contain a file named instances_default.json
find pre-pro -name '*.json' | head -3
# if they are named differently:
# find pre-pro -mindepth 2 -name '*.json' -exec sh -c 'mv "$1" "$(dirname "$1")/instances_default.json"' _ {} \;

# convert to YOLO. Run the script directly: it only needs ultralytics + pyyaml,
# so the flat-bug package (which requires Python >= 3.11) does NOT need installing.
python /path/to/flat-bug-1.0.0/src/bin/fb_prepare_data.py -i pre-pro -o fb_yolo -p 0.15 -f
```

**Current state (data prep and Stage 1 are already done on the server):**

1. Run the **CONFIG** cell.
2. Run the **Stage 2** cell (section 3). It is self-contained.
3. Run the **evaluation / metrics** cells.

Skip section 1 (converter) and section 2 (Stage 1) — already complete.

> After any CUDA device-side assert you must **restart the kernel** before retrying; the CUDA context stays corrupted otherwise.


In [ ]:
# ============================================================
# SEPTEMBER RUN — RF-DETR two-stage, whole tray @1536
# ============================================================
# TWO-STAGE: stage 1 pretrains on the REAL flat-bug photographs as a 1-class insect
# localiser; stage 2 fine-tunes that on the mixed 22-class trays. Roughly double the
# GPU time of a single-stage run.
#
# Its premise is worth re-checking before you spend that. The approach exists because
# flatbug beat RF-DETR on localisation (mAP@75 0.468 vs 0.340) — but that was measured
# against ground-truth boxes 1.295x too large on a side, and mAP@75 is exactly the
# metric a systematic box-size bias distorts. If the September tables show the two
# architectures close on mAP@75, the gap this was built to close does not exist.
#
# Everything else matches the other September notebooks: 22 classes, trays at
# 3473x2769 (10.85 px/mm, same as real), boxes that are the insect rather than the
# rotated canvas, and evaluation on 3dtrab_coco_TESTSPLIT.json.

# ============================================================
# CONFIG — two-stage RF-DETR   <- only this cell needs editing
# ============================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # two-stage runs after the first four
from pathlib import Path

BASE_DIR = Path(os.environ.get("MA_BASE", "/data/MA_Atul_Ares"))

# ── Stage 1: REAL flat-bug data (1-class 'insect' detection) ──
FB_YOLO_YAML     = str(BASE_DIR / "fb_yolo" / "data.yaml")   # from fb_prepare_data.py
FLATBUG_COCO_DIR = str(BASE_DIR / "flatbug_coco")            # YOLO -> COCO output (written below)
STAGE1_EPOCHS    = 50

# ── Stage 2: synthetic 23-class trays ──
DATASET_DIR = str(BASE_DIR / "insect_dataset_mixed")   # synthetic + real, 22 classes

# ── Outputs: NEW folders so the single-stage baseline (0.3476) is not overwritten ──
OUTPUT_DIR        = str(BASE_DIR / "sept_run" / "rfdetr_twostage_1536")
STAGE1_OUTPUT_DIR = OUTPUT_DIR + "_stage1_flatbug"

MODEL_VARIANT = "RFDETRLarge"
RESOLUTION    = 1536   # trays are 3473 px, so 0.44x: a real beetle lands at ~58 px
                       # instead of 30 at the old 800. Multiple of 32, as RF-DETR needs.

CLASS_NAMES = [
    # 22 classes: Siphonaptera and Raphidioptera dropped;
    # Diplopoda -> Polyxenida, Chilopoda -> Lithobiomorpha + Scutigeromorpha.
    "Lepidoptera",
    "Psocodea",
    "Odonata",
    "Zoraptera",
    "Phasmatodea",
    "Zygentoma",
    "Thysanoptera",
    "Strepsiptera",
    "Archaeognatha",
    "Dermaptera",
    "Diptera",
    "Coleoptera",
    "Hymenoptera",
    "Hemiptera",
    "Orthoptera",
    "Trichoptera",
    "Neuroptera",
    "Araneae",
    "Opiliones",
    "Polyxenida",
    "Lithobiomorpha",
    "Scutigeromorpha",
]

# ── Hyper-parameters: IDENTICAL to the single-stage baseline run ──
EPOCHS                   = 100
BATCH_SIZE               = 4
GRAD_ACCUM_STEPS         = 4
LR                       = 1e-4
LR_ENCODER               = 1.5e-4
LR_DROP                  = 50
WARMUP_EPOCHS            = 5
WEIGHT_DECAY             = 1e-4
MULTI_SCALE              = True
USE_EMA                  = False
CHECKPOINT_INTERVAL      = 10
EVAL_INTERVAL            = 1
RUN_TEST                 = True     # Stage 2 only; Stage 1 forces run_test=False (no test split)
EARLY_STOPPING           = True
EARLY_STOPPING_PATIENCE  = 20
EARLY_STOPPING_MIN_DELTA = 0.001
TENSORBOARD              = True
WANDB                    = False
NUM_WORKERS              = 8
PROGRESS_BAR             = "rich"
NUM_GPUS                 = 1
DDP_STRATEGY             = "ddp_notebook" if NUM_GPUS > 1 else "auto"

assert Path(FB_YOLO_YAML).exists(), (
    f"flat-bug YOLO data not found: {FB_YOLO_YAML}\n"
    "Download + prepare it first (see the markdown cell above).")
assert Path(DATASET_DIR).is_dir(), f"synthetic dataset not found: {DATASET_DIR}"
print(f"Stage 1 : {FB_YOLO_YAML}  ({STAGE1_EPOCHS} epochs)  -> {STAGE1_OUTPUT_DIR}")
print(f"Stage 2 : {DATASET_DIR}  ({EPOCHS} epochs)          -> {OUTPUT_DIR}")
print(f"Model   : {MODEL_VARIANT} @ {RESOLUTION}px | eff. batch "
      f"{BATCH_SIZE} x {GRAD_ACCUM_STEPS} x {NUM_GPUS} = {BATCH_SIZE*GRAD_ACCUM_STEPS*NUM_GPUS}")


In [ ]:
# ── Import RF-DETR ──
import sys
from pathlib import Path

_src = str(Path(".").resolve() / "rf-detr-1.7.1" / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRLarge
ModelClass = {"RFDETRNano": RFDETRNano, "RFDETRSmall": RFDETRSmall,
              "RFDETRMedium": RFDETRMedium, "RFDETRLarge": RFDETRLarge}[MODEL_VARIANT]
print(f"{MODEL_VARIANT} ready.")


## 1. Convert the flat-bug YOLO data to COCO  ←  **RUN THIS FIRST**

The old `flatbug_coco/` was **deleted** because it was built with the broken polygon parse. This cell rebuilds it correctly from `fb_yolo` (which was never corrupt). Run the verification cell right after it before training anything.


In [ ]:
# ── Convert the flat-bug YOLO data -> COCO (1-class) for RF-DETR ──
import json, shutil
from pathlib import Path
from PIL import Image

_IMG = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def _load_yaml(p):
    try:
        import yaml
        return yaml.safe_load(open(p))
    except Exception:
        # minimal fallback parser for simple key: value / list yaml
        cfg = {}
        for ln in Path(p).read_text().splitlines():
            ln = ln.split("#", 1)[0].rstrip()
            if not ln or ":" not in ln:
                continue
            k, v = ln.split(":", 1); k = k.strip(); v = v.strip()
            if v.startswith("[") and v.endswith("]"):
                cfg[k] = [x.strip().strip("'\"") for x in v[1:-1].split(",") if x.strip()]
            elif v:
                cfg[k] = v.strip("'\"")
        return cfg

def _img2label(img_path):
    """Ultralytics rule: replace the last '/images/' with '/labels/', ext -> .txt."""
    p = Path(img_path)
    parts = list(p.parts)
    if "images" in parts:
        i = len(parts) - 1 - parts[::-1].index("images")
        parts[i] = "labels"
        return Path(*parts).with_suffix(".txt")
    return p.parent.parent / "labels" / (p.stem + ".txt")   # fallback: sibling labels/

def _resolve_images(root, spec):
    """spec may be a dir, a .txt list file, or a list of those."""
    specs = spec if isinstance(spec, list) else [spec]
    imgs = []
    for s in specs:
        sp = (root / s) if not Path(s).is_absolute() else Path(s)
        if sp.is_dir():
            imgs += [q for q in sp.rglob("*") if q.suffix.lower() in _IMG]
        elif sp.suffix.lower() == ".txt" and sp.exists():
            for ln in sp.read_text().splitlines():
                ln = ln.strip()
                if not ln:
                    continue
                q = Path(ln)
                if not q.is_absolute():
                    q = (root / q).resolve()
                imgs.append(q)
        elif sp.suffix.lower() in _IMG:
            imgs.append(sp)
    return sorted(set(imgs))

def _yolo_line_to_xyxy_norm(tokens):
    """Return (cls, x1, y1, x2, y2) NORMALISED, handling BOTH ultralytics label formats:
      * detection: `cls cx cy w h`                      (5 tokens)
      * segmentation: `cls x1 y1 x2 y2 x3 y3 ...`       (polygon; what convert_coco
        writes with use_segments=True, e.g. the flat-bug data) -> take the polygon's
        min/max extent as the box.
    Reading a polygon line as `cx cy w h` silently produces enormous wrong boxes."""
    v = [float(x) for x in tokens[1:]]
    c = int(float(tokens[0]))
    if len(v) == 4:
        cx, cy, w, h = v
        return c, cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2
    if len(v) >= 6 and len(v) % 2 == 0:
        xs, ys = v[0::2], v[1::2]
        return c, min(xs), min(ys), max(xs), max(ys)
    return None

def yolo_to_coco(data_yaml, out_dir, split_map=(("train", "train"), ("val", "valid"), ("test", "test")),
                 link=True):
    data_yaml = Path(data_yaml); out_dir = Path(out_dir)
    cfg = _load_yaml(data_yaml)
    root = Path(cfg.get("path", data_yaml.parent))
    if not root.is_absolute():
        root = (data_yaml.parent / root).resolve()
    names = cfg.get("names")
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names, key=lambda x: int(x))]
    if not names:
        names = ["insect"]
    categories = [{"id": i + 1, "name": n, "supercategory": "none"} for i, n in enumerate(names)]
    summary = {}
    for ysplit, csplit in split_map:
        if cfg.get(ysplit) is None:
            continue
        img_paths = _resolve_images(root, cfg[ysplit])
        if not img_paths:
            continue
        sdir = out_dir / csplit
        if sdir.exists():
            shutil.rmtree(sdir)
        sdir.mkdir(parents=True)
        images, annotations, ann_id = [], [], 1
        for img_id, ip in enumerate(img_paths, 1):
            try:
                with Image.open(ip) as im:
                    W, H = im.size
            except Exception:
                continue
            out_name = f"{img_id:07d}_{ip.name}"
            dst = sdir / out_name
            try:
                if link:
                    dst.symlink_to(ip.resolve())
                else:
                    shutil.copy(ip, dst)
            except Exception:
                shutil.copy(ip, dst)
            images.append({"id": img_id, "file_name": out_name, "width": W, "height": H})
            lp = _img2label(ip)
            if lp.exists():
                for ln in lp.read_text().splitlines():
                    t = ln.split()
                    if len(t) < 5:
                        continue
                    parsed = _yolo_line_to_xyxy_norm(t)
                    if parsed is None:
                        continue
                    c, nx1, ny1, nx2, ny2 = parsed
                    x = nx1 * W; y = ny1 * H; w = (nx2 - nx1) * W; h = (ny2 - ny1) * H
                    x = max(0, min(x, W - 1)); y = max(0, min(y, H - 1))
                    w = max(1, min(w, W - x)); h = max(1, min(h, H - y))
                    annotations.append({"id": ann_id, "image_id": img_id,
                                        "category_id": c + 1, "bbox": [x, y, w, h],
                                        "area": float(w * h), "iscrowd": 0})
                    ann_id += 1
        coco = {"info": {"description": f"YOLO->COCO ({csplit})"},
                "categories": categories, "images": images, "annotations": annotations}
        json.dump(coco, open(sdir / "_annotations.coco.json", "w"))
        summary[csplit] = (len(images), len(annotations))
    return names, summary

_names, _summary = yolo_to_coco(FB_YOLO_YAML, FLATBUG_COCO_DIR, link=True)
print("flat-bug COCO classes:", _names)
for _s, (_ni, _na) in _summary.items():
    print(f"  {_s:6s}: {_ni} images, {_na} annotations")


In [ ]:
# ── VERIFY the rebuilt flatbug_coco BEFORE training (this check is what was missing) ──
import json, numpy as np
_a = json.load(open(f"{FLATBUG_COCO_DIR}/train/_annotations.coco.json"))
_wh = {i["id"]: (i["width"], i["height"]) for i in _a["images"]}
_fr = np.array([[b["bbox"][2] / _wh[b["image_id"]][0], b["bbox"][3] / _wh[b["image_id"]][1]]
                for b in _a["annotations"]])
print(f"{len(_a['images'])} images | {len(_fr)} boxes")
print(f"median box: {np.median(_fr[:,0])*100:.1f}% wide x {np.median(_fr[:,1])*100:.1f}% tall")
print(f"boxes covering >50% of the image: {(_fr.max(1) > 0.5).mean()*100:.1f}%")
assert np.median(_fr) < 0.15, (
    "BOXES FAR TOO LARGE - the polygon parse is wrong again. DO NOT TRAIN. "
    "(the old broken conversion gave ~27% x 80%)")
print("\nOK - boxes look like insects, safe to train.")


In [ ]:
# ── Sanity-check the converted flat-bug COCO dataset ──
import json as _json
for _split in ["train", "valid"]:
    _p = Path(FLATBUG_COCO_DIR) / _split / "_annotations.coco.json"
    if not _p.exists():
        print(f"  {_split:5s}: MISSING (run the converter cell above)"); continue
    _a = _json.load(open(_p))
    print(f"  {_split:5s}: {len(_a['images']):6d} images | {len(_a['annotations']):7d} annotations "
          f"| categories {[c['name'] for c in _a['categories']]}")
print("\nRF-DETR reads the class count from train/_annotations.coco.json -> should be 1 ('insect').")


## 2. Stage 1 — real-image pretraining  ←  **RUN THIS**

The previous Stage-1 checkpoint was trained on corrupted boxes and has been deleted, so this must be retrained from scratch.


In [ ]:
# ============================================================
# STAGE 1 — pretrain on the REAL flat-bug images (1-class detection)
# ============================================================
# Gives the model a real-world localisation prior. run_test=False because
# fb_prepare_data produces only train/val (no test split).
m1 = ModelClass(resolution=RESOLUTION)
m1.train(
    dataset_dir=FLATBUG_COCO_DIR, output_dir=STAGE1_OUTPUT_DIR, class_names=["insect"],
    epochs=STAGE1_EPOCHS, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=LR, lr_encoder=LR_ENCODER, lr_drop=STAGE1_EPOCHS, warmup_epochs=WARMUP_EPOCHS,
    weight_decay=WEIGHT_DECAY, multi_scale=MULTI_SCALE, use_ema=USE_EMA,
    checkpoint_interval=CHECKPOINT_INTERVAL, eval_interval=EVAL_INTERVAL,
    run_test=False,
    early_stopping=EARLY_STOPPING, early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA, progress_bar=PROGRESS_BAR,
    tensorboard=TENSORBOARD, wandb=WANDB, num_workers=NUM_WORKERS,
    devices=NUM_GPUS, strategy=DDP_STRATEGY, resume=None,
)
STAGE1_CKPT = str(Path(STAGE1_OUTPUT_DIR) / "checkpoint_best_total.pth")
assert Path(STAGE1_CKPT).exists(), f"Stage-1 checkpoint not found: {STAGE1_CKPT}"
print(f"\nStage-1 checkpoint (real-arthropod localiser): {STAGE1_CKPT}")


In [ ]:
# ── STAGE 1 (continued) — resume and train to a higher total epoch count ──
from pathlib import Path

STAGE1_EPOCHS_TOTAL = 100      # TOTAL target, not "50 more"

_d = Path(STAGE1_OUTPUT_DIR)
_cands = [_d / "last.ckpt"] + sorted(_d.glob("checkpoint_epoch=*.ckpt"))
_resume = next((str(p) for p in _cands if p.exists()), None)
assert _resume, f"no resumable .ckpt in {_d}. Found: {[p.name for p in _d.iterdir()]}"
print(f"resuming Stage 1 from {_resume} -> training to epoch {STAGE1_EPOCHS_TOTAL}")

m1 = ModelClass(resolution=RESOLUTION)
m1.train(
    dataset_dir=FLATBUG_COCO_DIR, output_dir=STAGE1_OUTPUT_DIR, class_names=["insect"],
    epochs=STAGE1_EPOCHS_TOTAL, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=LR, lr_encoder=LR_ENCODER, lr_drop=STAGE1_EPOCHS_TOTAL, warmup_epochs=WARMUP_EPOCHS,
    weight_decay=WEIGHT_DECAY, multi_scale=MULTI_SCALE, use_ema=USE_EMA,
    checkpoint_interval=CHECKPOINT_INTERVAL, eval_interval=EVAL_INTERVAL,
    run_test=False,
    early_stopping=EARLY_STOPPING, early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA, progress_bar=PROGRESS_BAR,
    tensorboard=TENSORBOARD, wandb=WANDB, num_workers=NUM_WORKERS,
    devices=NUM_GPUS, strategy=DDP_STRATEGY,
    resume=_resume,
)
STAGE1_CKPT = str(Path(STAGE1_OUTPUT_DIR) / "checkpoint_best_total.pth")
print(f"\nStage-1 (extended) best checkpoint: {STAGE1_CKPT}")


## 3. Stage 2 — synthetic fine-tuning

Run **after** Stage 1 finishes. `num_classes` is passed explicitly (without it the run dies with a CUDA device-side assert).


In [ ]:
# ============================================================
# STAGE 2 — fine-tune on the synthetic 23-class trays
# ============================================================
# SELF-CONTAINED: run the CONFIG cell, then THIS cell.
# The converter and Stage 1 are already done on disk and must NOT be re-run.
import sys
from pathlib import Path

_src = str(Path(".").resolve() / "rf-detr-1.7.1" / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)
from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRLarge
ModelClass = {"RFDETRNano": RFDETRNano, "RFDETRSmall": RFDETRSmall,
              "RFDETRMedium": RFDETRMedium, "RFDETRLarge": RFDETRLarge}[MODEL_VARIANT]

# Stage-1 prior (already trained; this cell does not re-run Stage 1)
STAGE1_CKPT = str(Path(STAGE1_OUTPUT_DIR) / "checkpoint_best_total.pth")
assert Path(STAGE1_CKPT).exists(), f"Stage-1 checkpoint not found: {STAGE1_CKPT}"
print(f"Stage-1 prior : {STAGE1_CKPT}")
print(f"Stage-2 data  : {DATASET_DIR}")
print(f"Stage-2 output: {OUTPUT_DIR}")

# ── CRITICAL: num_classes MUST be passed explicitly ──────────────────────────
# RFDETR.__init__ -> get_model() -> _build_model_context() -> load_pretrain_weights(),
# which MUTATES model_config.num_classes. Left at the default, it auto-aligns to the
# Stage-1 checkpoint (1 class) and marks the field as user-set, so the later
# dataset-based alignment refuses to raise it to 23. Training then feeds labels 0..22
# into a 1-class head -> CUDA device-side assert. Passing it here makes it a real
# override: the head is sized to the checkpoint for loading, then expanded to 23+1,
# while the backbone and bbox_embed keep the Stage-1 localisation prior.
model = ModelClass(resolution=RESOLUTION, pretrain_weights=STAGE1_CKPT,
                   num_classes=len(CLASS_NAMES))
assert model.model_config.num_classes == len(CLASS_NAMES), (
    f"num_classes is {model.model_config.num_classes}, expected {len(CLASS_NAMES)} - do NOT train")
print(f"num_classes   = {model.model_config.num_classes}  (detection head -> {len(CLASS_NAMES)}+1)")

model.train(
    dataset_dir=DATASET_DIR, output_dir=OUTPUT_DIR, class_names=CLASS_NAMES,
    epochs=EPOCHS, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=LR, lr_encoder=LR_ENCODER, lr_drop=LR_DROP, warmup_epochs=WARMUP_EPOCHS,
    weight_decay=WEIGHT_DECAY, multi_scale=MULTI_SCALE, use_ema=USE_EMA,
    checkpoint_interval=CHECKPOINT_INTERVAL, eval_interval=EVAL_INTERVAL, run_test=RUN_TEST,
    early_stopping=EARLY_STOPPING, early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA, progress_bar=PROGRESS_BAR,
    tensorboard=TENSORBOARD, wandb=WANDB, num_workers=NUM_WORKERS,
    devices=NUM_GPUS, strategy=DDP_STRATEGY,
    resume=None,
)
print(f"\nStage-2 checkpoints in: {OUTPUT_DIR}")
print("Compare the printed test mAP@50:95 against the single-stage baseline 0.3476 "
      "(and mAP@75 against 0.3400).")


## 4. Evaluation — load best checkpoint, then full metrics


In [ ]:
# ── Load the Stage-2 best checkpoint (for the metrics section below) ──
from rfdetr import RFDETR
BEST_CHECKPOINT = Path(OUTPUT_DIR) / "checkpoint_best_total.pth"
if BEST_CHECKPOINT.exists():
    model = RFDETR.from_checkpoint(str(BEST_CHECKPOINT), resolution=RESOLUTION)  # from_checkpoint does NOT restore resolution (falls back to the 704 default)
    print(f"Loaded {BEST_CHECKPOINT}")
else:
    print("Checkpoint not found; if training just finished, `model` already holds the best weights.")


In [ ]:
# ============================================================
# METRICS TOOLKIT (self-contained; flatbug/Ultralytics-style plots)
# box_iou, match_image, compute_metrics, confusion_matrix,
# plot_confusion, plot_curves, plot_labels, plot_results
# ============================================================
import numpy as np, json, math
from pathlib import Path
import matplotlib.pyplot as plt

# ── geometry ──────────────────────────────────────────────────────────────────
def box_iou(a, b):
    """IoU matrix between xyxy boxes a:(N,4) b:(M,4)."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    area_a = (a[:, 2] - a[:, 0]).clip(0) * (a[:, 3] - a[:, 1]).clip(0)
    area_b = (b[:, 2] - b[:, 0]).clip(0) * (b[:, 3] - b[:, 1]).clip(0)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / np.clip(union, 1e-9, None)

# ── per-image matching (class-aware, greedy by IoU) → tp over IoU thresholds ──
def match_image(iou, pred_cls, gt_cls, thresholds):
    """Return tp (Np, T) boolean: pred i is a TP at threshold t."""
    Np = len(pred_cls); T = len(thresholds)
    tp = np.zeros((Np, T), bool)
    if iou.size == 0 or len(gt_cls) == 0:
        return tp
    same = pred_cls[:, None] == gt_cls[None, :]
    for ti, thr in enumerate(thresholds):
        ci, gi = np.where((iou >= thr) & same)
        if not len(ci):
            continue
        order = np.argsort(-iou[ci, gi])
        ci, gi = ci[order], gi[order]
        used_g, used_p = set(), set()
        for c, g in zip(ci, gi):
            if c in used_p or g in used_g:
                continue
            used_p.add(c); used_g.add(g); tp[c, ti] = True
    return tp

# ── AP + curves (COCO-style 101-pt interp; curves vs confidence) ──────────────
# NumPy 2.0 removed np.trapz in favour of np.trapezoid. The training envs run
# different NumPy majors (flatbug torch 2.13 / rfdetr torch 2.9), so bind once
# and use _TRAPZ rather than depending on which one is installed.
_TRAPZ = getattr(np, "trapezoid", None) or np.trapz
def _ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([1.0], precision, [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101)
    return float(_TRAPZ(np.interp(x, mrec, mpre), x))

def compute_metrics(tp, conf, pred_cls, target_cls, nc, thresholds):
    """tp:(N,T). Returns dict of arrays + curves (Ultralytics layout)."""
    order = np.argsort(-conf)
    tp, conf, pred_cls = tp[order], conf[order], pred_cls[order]
    px = np.linspace(0, 1, 1000)
    T = tp.shape[1]
    ap = np.zeros((nc, T)); p_curve = np.zeros((nc, 1000)); r_curve = np.zeros((nc, 1000))
    f1_curve = np.zeros((nc, 1000)); prec_at = np.zeros((nc, 1000))  # precision vs recall(px)
    ap50 = np.zeros(nc); ap5095 = np.zeros(nc)
    n_gt = np.bincount(target_cls.astype(int), minlength=nc)
    n_p = np.bincount(pred_cls.astype(int), minlength=nc)
    for c in range(nc):
        i = pred_cls == c
        if n_gt[c] == 0 and i.sum() == 0:
            continue
        if i.sum() == 0:
            continue
        fpc = (1 - tp[i]).cumsum(0)
        tpc = tp[i].cumsum(0)
        recall = tpc / (n_gt[c] + 1e-16)
        precision = tpc / (tpc + fpc + 1e-16)
        # curves vs confidence threshold px (conf of this class's preds, desc)
        cc = conf[i]
        r_curve[c] = np.interp(-px, -cc, recall[:, 0], left=0)
        p_curve[c] = np.interp(-px, -cc, precision[:, 0], left=1)
        f1_curve[c] = 2 * p_curve[c] * r_curve[c] / (p_curve[c] + r_curve[c] + 1e-16)
        # precision vs recall (for PR_curve), at IoU 0.5 (threshold index 0)
        prec_at[c] = np.interp(px, recall[:, 0], precision[:, 0], left=1)
        for ti in range(T):
            ap[c, ti] = _ap(recall[:, ti], precision[:, ti])
        ap50[c] = ap[c, 0]; ap5095[c] = ap[c].mean()
    return dict(ap=ap, ap50=ap50, ap5095=ap5095, p_curve=p_curve, r_curve=r_curve,
                f1_curve=f1_curve, prec_at=prec_at, px=px, n_gt=n_gt, n_p=n_p)

# ── confusion matrix (fixed conf+iou, class-agnostic assignment) ──────────────
def confusion_matrix(images, nc, conf_thres=0.25, iou_thres=0.45):
    """images: list of dicts with pred_xyxy,pred_conf,pred_cls,gt_xyxy,gt_cls.
    matrix[pred, true], last row/col = background (FP row / FN col)."""
    cm = np.zeros((nc + 1, nc + 1), dtype=np.int64)
    for im in images:
        pc, pcf, pcl = im["pred_xyxy"], im["pred_conf"], im["pred_cls"]
        keep = pcf >= conf_thres
        pc, pcl = pc[keep], pcl[keep]
        gc, gcl = im["gt_xyxy"], im["gt_cls"]
        if len(gcl) == 0:
            for c in pcl:
                cm[int(c), nc] += 1            # predicted, no GT → FP (background col)
            continue
        if len(pcl) == 0:
            for c in gcl:
                cm[nc, int(c)] += 1            # GT missed → FN (background row)
            continue
        iou = box_iou(pc, gc)
        ci, gi = np.where(iou >= iou_thres)
        matched_p, matched_g = set(), set()
        if len(ci):
            order = np.argsort(-iou[ci, gi])
            for c, g in zip(ci[order], gi[order]):
                if c in matched_p or g in matched_g:
                    continue
                matched_p.add(c); matched_g.add(g)
                cm[int(pcl[c]), int(gcl[g])] += 1
        for c in range(len(pcl)):
            if c not in matched_p:
                cm[int(pcl[c]), nc] += 1        # FP
        for g in range(len(gcl)):
            if g not in matched_g:
                cm[nc, int(gcl[g])] += 1        # FN
    return cm

# ── plots ─────────────────────────────────────────────────────────────────────
def plot_confusion(cm, names, save, normalize=False):
    m = cm.astype(float)
    if normalize:
        m = m / (m.sum(0, keepdims=True) + 1e-9)   # column-normalize (per true class)
    labels = list(names) + ["background"]
    n = len(labels)
    fig, ax = plt.subplots(figsize=(max(9, n * 0.55), max(8, n * 0.5)))
    im = ax.imshow(m, cmap="Blues", vmin=0, vmax=(1.0 if normalize else None))
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel("True"); ax.set_ylabel("Predicted")
    ax.set_title("Confusion Matrix" + (" (normalized)" if normalize else ""))
    thr = (m.max() or 1) * 0.5
    if n <= 30:
        for i in range(n):
            for j in range(n):
                v = m[i, j]
                if v > 0:
                    ax.text(j, i, f"{v:.2f}" if normalize else f"{int(v)}",
                            ha="center", va="center", fontsize=6,
                            color="white" if v > thr else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout(); fig.savefig(save, dpi=200); plt.close(fig)

def _curve(x, ys, names, xlabel, ylabel, title, save, mean_label):
    fig, ax = plt.subplots(figsize=(9, 6))
    ys = np.asarray(ys)
    for i, y in enumerate(ys):
        ax.plot(x, y, linewidth=1, label=names[i] if len(names) <= 25 else None)
    ax.plot(x, ys.mean(0), linewidth=3, color="navy", label=mean_label)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.01); ax.grid(alpha=0.3)
    if len(names) <= 25:
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
    fig.tight_layout(); fig.savefig(save, dpi=200); plt.close(fig)

def plot_curves(M, names, outdir):
    px = M["px"]
    _curve(px, M["prec_at"], names, "Recall", "Precision",
           f"Precision-Recall  (mAP@0.5={np.nan_to_num(M['ap50']).mean():.3f})",
           outdir / "PR_curve.png", f"all classes {np.nan_to_num(M['ap50']).mean():.3f} mAP@0.5")
    f1 = M["f1_curve"]; f1m = f1.mean(0); bi = int(f1m.argmax())
    _curve(px, f1, names, "Confidence", "F1",
           f"F1-Confidence  (best {f1m[bi]:.2f} @ {px[bi]:.2f})",
           outdir / "F1_curve.png", f"all classes {f1m[bi]:.2f} @ {px[bi]:.2f}")
    _curve(px, M["p_curve"], names, "Confidence", "Precision", "Precision-Confidence",
           outdir / "P_curve.png", "all classes")
    _curve(px, M["r_curve"], names, "Confidence", "Recall", "Recall-Confidence",
           outdir / "R_curve.png", "all classes")

def plot_labels(gt_cls_all, gt_xywh_norm, names, save):
    """gt_xywh_norm: (M,4) cx,cy,w,h normalized [0,1]."""
    fig, ax = plt.subplots(2, 2, figsize=(12, 10))
    nc = len(names)
    cnt = np.bincount(gt_cls_all.astype(int), minlength=nc)
    ax[0, 0].bar(range(nc), cnt, color="steelblue")
    ax[0, 0].set_xticks(range(nc)); ax[0, 0].set_xticklabels(names, rotation=90, fontsize=7)
    ax[0, 0].set_title("instances per class"); ax[0, 0].set_ylabel("count")
    g = gt_xywh_norm
    if len(g):
        ax[0, 1].hist2d(g[:, 0], g[:, 1], bins=50, cmap="viridis")
        ax[0, 1].set_title("box center  x-y"); ax[0, 1].set_xlabel("x"); ax[0, 1].set_ylabel("y")
        ax[0, 1].set_xlim(0, 1); ax[0, 1].set_ylim(1, 0)
        ax[1, 0].hist2d(g[:, 2], g[:, 3], bins=50, cmap="viridis")
        ax[1, 0].set_title("box  width-height"); ax[1, 0].set_xlabel("width"); ax[1, 0].set_ylabel("height")
        asp = g[:, 2] / np.clip(g[:, 3], 1e-6, None)
        ax[1, 1].hist(np.clip(asp, 0, 5), bins=60, color="steelblue")
        ax[1, 1].set_title("aspect ratio (w/h)")
    fig.tight_layout(); fig.savefig(save, dpi=150); plt.close(fig)

def plot_results(metrics_csv, save):
    """Parse RF-DETR's Lightning metrics.csv → grid of metric-vs-epoch curves."""
    import pandas as pd
    df = pd.read_csv(metrics_csv)
    epoch_col = "epoch" if "epoch" in df.columns else ("step" if "step" in df.columns else None)
    if epoch_col is None:
        epoch_col = "_row"
        df[epoch_col] = np.arange(len(df))
    num = [c for c in df.columns if c not in (epoch_col, "step") and
           np.issubdtype(df[c].dtype, np.number)]
    # collapse to one row per epoch (mean of non-NaN) so train/val logs align
    g = df.groupby(epoch_col)[num].mean(numeric_only=True)
    cols = [c for c in num if g[c].notna().any()]
    if not cols:
        return False
    ncol = min(4, len(cols)); nrow = math.ceil(len(cols) / ncol)
    fig, ax = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3 * nrow), squeeze=False)
    for k, c in enumerate(cols):
        a = ax[k // ncol][k % ncol]
        s = g[c].dropna()
        a.plot(s.index, s.values, marker=".", ms=3, lw=1.2)
        a.set_title(c, fontsize=9); a.set_xlabel(epoch_col); a.grid(alpha=0.3)
    for k in range(len(cols), nrow * ncol):
        ax[k // ncol][k % ncol].axis("off")
    fig.suptitle("Training results", fontsize=13)
    fig.tight_layout(); fig.savefig(save, dpi=150); plt.close(fig)
    return True


In [ ]:
# ============================================================
# DIAGNOSTICS — run predictions on the eval split + collect GT
# ============================================================
import numpy as np, json, math
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

EVAL_SPLIT   = "test"            # "test" or "valid"
PRED_CONF_LOW = 0.05             # low threshold -> full PR curve (filter later)
CM_CONF, CM_IOU = 0.25, 0.45     # confusion-matrix operating point (YOLO defaults)
IOU_VEC = np.linspace(0.5, 0.95, 10)   # COCO IoU sweep for mAP@0.5:0.95

METRICS_DIR = Path(OUTPUT_DIR) / f"metrics_{EVAL_SPLIT}"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
NC = len(CLASS_NAMES)
_name_to_idx = {n: i for i, n in enumerate(CLASS_NAMES)}

# ── ground truth (COCO) ──
_ann = json.load(open(Path(DATASET_DIR) / EVAL_SPLIT / "_annotations.coco.json"))
_imgs_meta = {im["id"]: im for im in _ann["images"]}
_catname   = {c["id"]: c["name"] for c in _ann["categories"]}
_gt = {}
for a in _ann["annotations"]:
    _ci = _name_to_idx.get(_catname.get(a["category_id"]))
    if _ci is None:
        continue
    x, y, w, h = a["bbox"]
    _gt.setdefault(a["image_id"], []).append(([x, y, x + w, y + h], _ci))

# ── predict + collect ──
_images = []; _tp = []; _conf = []; _pcls = []; _tcls = []; _gt_cls_all = []; _gt_xywh = []
for _iid, _im in tqdm(_imgs_meta.items(), desc=f"Predict {EVAL_SPLIT}"):
    _W, _H = _im["width"], _im["height"]
    _pil = Image.open(Path(DATASET_DIR) / EVAL_SPLIT / _im["file_name"]).convert("RGB")
    _d = model.predict(_pil, threshold=PRED_CONF_LOW)
    if len(_d) > 0:
        _px = np.asarray(_d.xyxy, float).reshape(-1, 4)
        _pc = np.asarray(_d.confidence, float)
        _names = _d.data.get("class_name", None)
        if _names is not None and len(_names) == len(_pc):
            _pl = np.array([_name_to_idx.get(str(n), -1) for n in _names])
        else:
            _pl = np.asarray(_d.class_id, int)
            if _pl.size and _pl.max() >= NC:   # 1-indexed COCO fallback
                _pl = _pl - 1
        _k = _pl >= 0
        _px, _pc, _pl = _px[_k], _pc[_k], _pl[_k]
    else:
        _px = np.zeros((0, 4)); _pc = np.zeros(0); _pl = np.zeros(0, int)
    _g = _gt.get(_iid, [])
    _gx = np.array([b for b, _ in _g], float).reshape(-1, 4)
    _gc = np.array([c for _, c in _g], int)
    _images.append(dict(pred_xyxy=_px, pred_conf=_pc, pred_cls=_pl, gt_xyxy=_gx, gt_cls=_gc))
    _tp.append(match_image(box_iou(_px, _gx), _pl, _gc, IOU_VEC))
    _conf.append(_pc); _pcls.append(_pl); _tcls.append(_gc); _gt_cls_all.append(_gc)
    for x1, y1, x2, y2 in _gx:
        _gt_xywh.append([((x1 + x2) / 2) / _W, ((y1 + y2) / 2) / _H, (x2 - x1) / _W, (y2 - y1) / _H])

_tp   = np.concatenate(_tp) if _tp else np.zeros((0, len(IOU_VEC)), bool)
_conf = np.concatenate(_conf) if _conf else np.zeros(0)
_pcls = np.concatenate(_pcls) if _pcls else np.zeros(0, int)
_tcls = np.concatenate(_tcls) if _tcls else np.zeros(0, int)
print(f"Collected {len(_conf)} predictions, {len(_tcls)} GT instances over {len(_images)} images "
      f"({EVAL_SPLIT}).")


In [ ]:
# ============================================================
# DIAGNOSTICS — compute metrics + write all flatbug-style outputs
# ============================================================
import pandas as pd, shutil

_M  = compute_metrics(_tp, _conf, _pcls, _tcls, NC, IOU_VEC)
_cm = confusion_matrix(_images, NC, conf_thres=CM_CONF, iou_thres=CM_IOU)

plot_confusion(_cm, CLASS_NAMES, METRICS_DIR / "confusion_matrix.png", normalize=False)
plot_confusion(_cm, CLASS_NAMES, METRICS_DIR / "confusion_matrix_normalized.png", normalize=True)
plot_curves(_M, CLASS_NAMES, METRICS_DIR)
plot_labels(np.concatenate(_gt_cls_all) if _gt_cls_all else np.zeros(0, int),
            np.array(_gt_xywh).reshape(-1, 4), CLASS_NAMES, METRICS_DIR / "labels.jpg")

# training curves from RF-DETR's Lightning metrics.csv
_mcsv = Path(OUTPUT_DIR) / "metrics.csv"
if _mcsv.exists():
    shutil.copy(_mcsv, METRICS_DIR / "results.csv")
    plot_results(_mcsv, METRICS_DIR / "results.png")
else:
    print(f"(no metrics.csv in {OUTPUT_DIR} - skipping results.png)")

# per-class table (+ mean P/R at best-F1 conf, like the YOLO printout)
_f1m = _M["f1_curve"].mean(0); _bi = int(_f1m.argmax())
# AP@75 per class. _M["ap"] holds all 10 IoU thresholds (0.50..0.95 step 0.05), so
# column 5 IS AP75 — it was being computed and never written. The tiled notebooks
# report per-class AP75, so without this the comparison has a hole on the whole-tray
# row, and AP75 is the metric the two-stage experiment is built around.
_ap75 = _M["ap"][:, 5] if "ap" in _M and _M["ap"].ndim == 2 and _M["ap"].shape[1] > 5 else None
_rows = [dict(**{"class": n}, instances=int(_M["n_gt"][i]), preds=int(_M["n_p"][i]),
              P=round(float(_M["p_curve"][i, _bi]), 4), R=round(float(_M["r_curve"][i, _bi]), 4),
              mAP50=round(float(_M["ap50"][i]), 4),
              mAP75=(round(float(_ap75[i]), 4) if _ap75 is not None else None),
              mAP50_95=round(float(_M["ap5095"][i]), 4))
         for i, n in enumerate(CLASS_NAMES)]
_all = dict(**{"class": "ALL"}, instances=int(_M["n_gt"].sum()), preds=int(_M["n_p"].sum()),
            P=round(float(_M["p_curve"][:, _bi].mean()), 4), R=round(float(_M["r_curve"][:, _bi].mean()), 4),
            mAP50=round(float(np.nan_to_num(_M["ap50"]).mean()), 4),
            mAP75=(round(float(np.nan_to_num(_ap75).mean()), 4) if _ap75 is not None else None),
            mAP50_95=round(float(np.nan_to_num(_M["ap5095"]).mean()), 4))
_pcdf = pd.DataFrame(_rows + [_all])
_pcdf.to_csv(METRICS_DIR / "per_class_metrics.csv", index=False)

print("\n=== EVAL METRICS (" + EVAL_SPLIT + ") ===")
print(f"  mAP50-95 : {_all['mAP50_95']:.4f}")
print(f"  mAP50    : {_all['mAP50']:.4f}")
if _all.get("mAP75") is not None:
    print(f"  mAP75    : {_all['mAP75']:.4f}")
print(f"  P (best-F1 conf {_M['px'][_bi]:.2f}) : {_all['P']:.4f}")
print(f"  R (best-F1 conf {_M['px'][_bi]:.2f}) : {_all['R']:.4f}")
print(_pcdf.to_string(index=False))
print(f"\nAll plots + CSVs saved to: {METRICS_DIR}")
for _f in sorted(METRICS_DIR.glob("*")):
    print("   ", _f.name)


In [ ]:
# ============================================================
# DIAGNOSTICS — val_batch previews (GT labels vs predictions)
# ============================================================
import random as _rnd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

_sample = _rnd.Random(0).sample(list(_imgs_meta), min(9, len(_imgs_meta)))
_palette = plt.cm.tab20(np.linspace(0, 1, NC))

for _tag, _use_pred in [("labels", False), ("pred", True)]:
    _c = 3; _r = math.ceil(len(_sample) / _c)
    fig, axes = plt.subplots(_r, _c, figsize=(5 * _c, 4 * _r))
    axes = np.array(axes).reshape(-1)
    for _ax, _iid in zip(axes, _sample):
        _im = _imgs_meta[_iid]
        _pil = Image.open(Path(DATASET_DIR) / EVAL_SPLIT / _im["file_name"]).convert("RGB")
        _ax.imshow(_pil); _ax.axis("off"); _ax.set_title(_im["file_name"][:24], fontsize=7)
        if _use_pred:
            _d = model.predict(_pil, threshold=CM_CONF)
            _nm = _d.data.get("class_name", []) if len(_d) > 0 else []
            for _bi2, ((x1, y1, x2, y2), cf) in enumerate(
                    zip(_d.xyxy, _d.confidence) if len(_d) > 0 else []):
                _ci = _name_to_idx.get(str(_nm[_bi2]), 0) if _bi2 < len(_nm) else 0
                _ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, lw=1,
                              edgecolor=_palette[_ci % NC], facecolor="none"))
        else:
            for (x1, y1, x2, y2), _ci in _gt.get(_iid, []):
                _ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, lw=1,
                              edgecolor=_palette[_ci % NC], facecolor="none"))
    for _ax in axes[len(_sample):]:
        _ax.axis("off")
    fig.suptitle(f"{EVAL_SPLIT}_batch_{_tag}", fontsize=13)
    fig.tight_layout()
    fig.savefig(METRICS_DIR / f"val_batch_{_tag}.jpg", dpi=130, bbox_inches="tight")
    plt.show()
print(f"Saved val_batch_labels.jpg / val_batch_pred.jpg -> {METRICS_DIR}")
